In [1]:
%load_ext autoreload
%autoreload 2

pattern = "pattern-12"

entrypoint = pattern
app_cwl_file = f"../cwl-workflow/{pattern}.cwl"

try:
    from docs.helpers import plot_cwl, wrap_cwl
except (ImportError, ModuleNotFoundError) as e:

    import os
    import sys

    module_path = os.path.abspath(os.path.join("."))  # or the path to your source code
    sys.path.insert(0, module_path)

from helpers import WorkflowViewer, WorkflowWrapper
from cwl_loader import dump_cwl
from IPython.display import Markdown, display
import json

In [2]:
wf = WorkflowViewer.from_file(app_cwl_file, entrypoint)

2025-12-01 16:13:27.345 | DEBUG    | cwl_loader:load_cwl_from_location:228 - Loading CWL document from ../cwl-workflow/pattern-12.cwl...
2025-12-01 16:13:27.346 | DEBUG    | cwl_loader:_load_cwl_from_stream:231 - Reading stream from ../cwl-workflow/pattern-12.cwl...
2025-12-01 16:13:27.370 | DEBUG    | cwl_loader:load_cwl_from_stream:203 - CWL data of type <class 'ruamel.yaml.comments.CommentedMap'> successfully loaded from stream
2025-12-01 16:13:27.371 | DEBUG    | cwl_loader:load_cwl_from_yaml:130 - Updating the model from version 'v1.0' to version 'v1.2'...
2025-12-01 16:13:27.371 | DEBUG    | cwl_loader:load_cwl_from_yaml:141 - Raw CWL document successfully updated to v1.2!
2025-12-01 16:13:27.371 | DEBUG    | cwl_loader:load_cwl_from_yaml:145 - Parsing the raw CWL document to the CWL Utils DOM...


ValidationException: Value is a array, but valid type for this field is an object.
Value is a array, but valid type for this field is an object.
Value is a array, but valid type for this field is an object.
Value is a array, but valid type for this field is an object.
Value is a array, but valid type for this field is an object.

## Application Package Pattern 12

This patterns publishes all products generated by all steps

In [ ]:
wf.plot()

### Inputs

In [ ]:
wf.display_inputs()

### Steps

In [ ]:
wf.display_steps()

### Outputs

In [ ]:
wf.display_outputs()

## Data flow management

In [ ]:
w = WorkflowWrapper(workflow=wf.workflow, entrypoint=entrypoint)
wrapped = w.wrap()

app_cwl_file = f".{entrypoint}.cwl"

with open(app_cwl_file, "w") as f:
    dump_cwl(process=wrapped, stream=f)

In [ ]:
wf = WorkflowViewer(cwl_file=app_cwl_file, workflow=wrapped, entrypoint="main")

wf.plot()

### Workflow components diagram

In [ ]:
wf.display_components_diagram()

### Inputs

In [ ]:
wf.display_inputs()

### Steps

In [ ]:
wf.display_steps()

### Outputs

In [ ]:
wf.display_outputs()

## Execution


In [ ]:
from cwltool.main import main
from io import StringIO
import argparse
import yaml

In [ ]:
params = {
    "aoi": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/ogc.yaml#BBox",
        "bbox": [-118.985, 38.432, -118.183, 38.938],
        "crs": "CRS84",
    },
    "item": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/ogc.yaml#URI",
        "value": "https://planetarycomputer.microsoft.com/api/stac/v1/collections/landsat-c2-l2/items/LC08_L2SP_042033_20231007_02_T1",
    },
    "bands": ["green", "nir08"],
    "cropped-collection": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/ogc.yaml#URI",
        "value": "https://github.com/eoap/application-package-patterns/raw/refs/heads/develop/collections/cropped_collection.json",
    },
    "ndwi-collection": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/ogc.yaml#URI",
        "value": "https://github.com/eoap/application-package-patterns/raw/refs/heads/develop/collections/ndi_collection.json",
    },
    "water-bodies-collection": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/ogc.yaml#URI",
        "value": "https://github.com/eoap/application-package-patterns/raw/refs/heads/develop/collections/water_bodies_collection.json",
    },
}

additional_params = {
    "another_input": "some_value",
    "s3_bucket": "my-bucket",
    "sub_path": "my/sub/path",
    "aws_access_key_id": "test",
    "aws_secret_access_key": "test",
    "region_name": "us-west-1",
    "endpoint_url": "https://s3.us-west-1.amazonaws.com",
}

with open(".params.yaml", "w") as f:
    yaml.dump({**params, **additional_params}, f)

md = f"""

### Inputs

```json
{json.dumps(params, indent=2)}
```
"""

display(Markdown(md))

In [ ]:
parsed_args = argparse.Namespace(
    podman=False,
    debug=False,
    validate=False,
    outdir="./runs",
    workflow=f"{app_cwl_file}#main",
    job_order=[".params.yaml"],
)

stream_out = StringIO()
stream_err = StringIO()

res = main(
    args=parsed_args,
    stdout=stream_out,
    stderr=stream_err,
)

if res == 0:
    md = f"""

### Outputs

```json
{stream_out.getvalue()}
```
    """
else:
    md = f"""

### Errors

CWL execution terminated with errors:

```
{stream_err.getvalue()}
```
    """

display(Markdown(md))